# Evaluation

This notebook contains the code used to sample from the WSI images and split them into a training and test set. 

In [ ]:
import sys
from pathlib import Path
import pyvips
import numpy as np
import cv2

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import numpy as np
import random
from scripts.filters import *
from scripts.evaluate import *
from scripts.utils import *

project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root))
print(project_root)
print(Path.cwd())

## Sample tiles from all tiff files

In [ ]:
output_dir = Path("./evaluation")
output_dir.mkdir(exist_ok=True)

path = "/media/sdog/SSD-500/microglia_data"
files = [f for f in Path(path).glob("./*.tiff")]
print(len(files))

In [ ]:
def sample_tiles(num_tiles, img, img_name, patch_size, out_path, save=True):
    patches = []
    i = 1
    it = 1
    max_iterations = 35 * num_tiles
    while i <= num_tiles and it < max_iterations:
        x = random.randint(0, img.width - patch_size)
        y = random.randint(0, img.height - patch_size)
        cropped = img.crop(x, y, patch_size, patch_size)
        patch_np = np.ndarray(
            buffer=cropped.write_to_memory(),
            dtype=np.uint8,
            shape=[cropped.height, cropped.width, cropped.bands],
        )
        if not filter_tissue(patch_np):
            if save:
                filename = f"{img_name}_{i}.png"
                cropped.write_to_file(str(out_path / filename))
            else:
                patches.append(cropped)
            i += 1
        it += 1

    if i < num_tiles:
        print(f"WARNING: only {i}/{num_tiles} patches saved after {it}")

    return patches

In [ ]:
for file in files:
    img = pyvips.Image.new_from_file(file, access="sequential")
    sample_tiles(10, img, file.name, 512, output_dir, save=False)

## Sort sampled tiles into test and training set randomly

In [ ]:
from sklearn.model_selection import train_test_split
import os
import shutil

src_dir = Path("./evaluation")
train_dir = src_dir / "train"
test_dir = src_dir / "test"
train_dir.mkdir(exist_ok=True)
test_dir.mkdir(exist_ok=True)
files = [f for f in src_dir.iterdir() if f.is_file()]
train, test = train_test_split(files, test_size=0.3, random_state=36)

In [ ]:
for file in train:
    shutil.copy(file, train_dir / file.name)

In [ ]:
for file in test:
    shutil.copy(file, test_dir / file.name)